# Mixed Precision Training

**Interview Focus**: FP32/FP16/BF16 tradeoffs, AMP implementation, loss scaling.

**Key Concepts**:
- FP32 vs FP16 vs BF16: precision, range, use cases
- `torch.cuda.amp`: autocast + GradScaler
- Loss scaling: why needed for FP16, not BF16
- Which ops are safe in lower precision

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import struct

## 1. Floating Point Formats

IEEE 754 layout: `[sign | exponent | mantissa]`

| Format | Bits | Exponent | Mantissa | Range | Precision |
|--------|------|----------|----------|-------|-----------|
| FP32 | 32 | 8 bits | 23 bits | ~1e-38 to 3e38 | ~7 decimal digits |
| FP16 | 16 | 5 bits | 10 bits | ~6e-8 to 65504 | ~3.3 decimal digits |
| BF16 | 16 | 8 bits | 7 bits | ~1e-38 to 3e38 | ~2.4 decimal digits |

**BF16 key insight**: Same exponent range as FP32 (8 bits), so no overflow/underflow issues. Less precise, but that's fine for gradients.

In [ ]:
def inspect_dtype(dtype, name):
    """Show properties of a floating-point dtype."""
    info = torch.finfo(dtype)
    print(f"{name}:")
    print(f"  Bits:         {info.bits}")
    print(f"  Max value:    {info.max}")
    print(f"  Min value:    {info.min}")
    print(f"  Smallest normal: {info.tiny}")
    print(f"  Epsilon:      {info.eps}")
    print()

inspect_dtype(torch.float32, "FP32")
inspect_dtype(torch.float16, "FP16")
inspect_dtype(torch.bfloat16, "BF16")

In [ ]:
# Demonstrate overflow in FP16
val = torch.tensor(70000.0, dtype=torch.float32)
print(f"FP32: {val.item()}")
print(f"FP16: {val.half().item()}")
print(f"BF16: {val.bfloat16().item()}")

print()

# Demonstrate precision loss
val = torch.tensor(1.0001, dtype=torch.float32)
print(f"FP32: {val.item():.6f}")
print(f"FP16: {val.half().item():.6f}")
print(f"BF16: {val.bfloat16().item():.6f}")

print()

# Demonstrate underflow: small gradients vanish in FP16
small_grad = torch.tensor(1e-8, dtype=torch.float32)
print(f"Small gradient in FP32: {small_grad.item()}")
print(f"Small gradient in FP16: {small_grad.half().item()}")   # becomes 0!
print(f"Small gradient in BF16: {small_grad.bfloat16().item()}")  # also 0, but loss scaling helps

## 2. Loss Scaling — Why It's Needed for FP16

**Problem**: Small gradient values (< 6e-8) underflow to zero in FP16.

**Solution**: Scale the loss up before backward pass, then scale gradients down before optimizer step.

```
scaled_loss = loss * scale_factor     # e.g., scale_factor = 1024
scaled_loss.backward()                # gradients are scaled up too
optimizer.step(grad / scale_factor)   # scale back down
```

**Dynamic loss scaling**: Start with a large scale (e.g., 2^16). If inf/nan in gradients, skip update and halve the scale. Otherwise, periodically increase the scale.

**BF16 doesn't need loss scaling** because it has the same exponent range as FP32 — same dynamic range, no underflow problem.

In [ ]:
class ManualGradScaler:
    """Simplified GradScaler to understand the mechanics."""
    
    def __init__(self, init_scale=2**16, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0
    
    def scale_loss(self, loss):
        """Multiply loss by scale factor before backward."""
        return loss * self.scale
    
    def unscale_and_step(self, optimizer, model):
        """Unscale gradients and step if no inf/nan."""
        # Unscale gradients
        for param in model.parameters():
            if param.grad is not None:
                param.grad /= self.scale
        
        # Check for inf/nan
        has_inf_nan = False
        for param in model.parameters():
            if param.grad is not None:
                if torch.isinf(param.grad).any() or torch.isnan(param.grad).any():
                    has_inf_nan = True
                    break
        
        if has_inf_nan:
            # Skip this step, reduce scale
            self.scale *= self.backoff_factor
            optimizer.zero_grad()  # discard these gradients
            return False
        else:
            optimizer.step()
            self.steps_since_growth += 1
            if self.steps_since_growth >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_growth = 0
            return True


print("Manual GradScaler implemented.")
print(f"Initial scale: {2**16} = 65536")
print("If inf/nan found: scale *= 0.5 (halve)")
print("Every 2000 good steps: scale *= 2.0 (double)")

## 3. Mixed Precision Training with `torch.cuda.amp`

The standard PyTorch AMP pattern. Works on CPU too (for demonstration), but speedup is only on GPU.

In [ ]:
from torch.amp import autocast, GradScaler

# Simple model for demonstration
class DemoNet(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, output_dim=10):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim),
        )
    
    def forward(self, x):
        return self.layers(x)

# Generate synthetic data
torch.manual_seed(42)
X_train = torch.randn(1000, 784)
y_train = torch.randint(0, 10, (1000,))

print("Model and data created.")

In [ ]:
def train_fp32(model, X, y, epochs=10):
    """Standard FP32 training."""
    model = model.float()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    losses = []
    start = time.time()
    
    for epoch in range(epochs):
        # Mini-batch training
        for i in range(0, len(X), 64):
            x_batch = X[i:i+64]
            y_batch = y[i:i+64]
            
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        losses.append(loss.item())
    
    elapsed = time.time() - start
    return losses, elapsed


def train_amp(model, X, y, epochs=10, dtype=torch.float16):
    """Mixed precision training with autocast and GradScaler."""
    model = model.float()  # master weights stay in FP32
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    
    # GradScaler is needed for FP16, optional for BF16
    use_scaler = (dtype == torch.float16)
    scaler = GradScaler('cpu', enabled=use_scaler)
    
    losses = []
    start = time.time()
    
    for epoch in range(epochs):
        for i in range(0, len(X), 64):
            x_batch = X[i:i+64]
            y_batch = y[i:i+64]
            
            optimizer.zero_grad()
            
            # autocast: eligible ops run in lower precision
            with autocast(device_type='cpu', dtype=dtype):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
            
            # GradScaler: scale loss, backward, unscale, step
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        
        losses.append(loss.item())
    
    elapsed = time.time() - start
    return losses, elapsed


# Run comparisons
torch.manual_seed(42)
model_fp32 = DemoNet()
losses_fp32, time_fp32 = train_fp32(DemoNet(), X_train, y_train, epochs=20)

torch.manual_seed(42)
losses_fp16, time_fp16 = train_amp(DemoNet(), X_train, y_train, epochs=20, dtype=torch.float16)

torch.manual_seed(42)
losses_bf16, time_bf16 = train_amp(DemoNet(), X_train, y_train, epochs=20, dtype=torch.bfloat16)

print(f"FP32:  final_loss={losses_fp32[-1]:.4f}  time={time_fp32:.3f}s")
print(f"FP16:  final_loss={losses_fp16[-1]:.4f}  time={time_fp16:.3f}s")
print(f"BF16:  final_loss={losses_bf16[-1]:.4f}  time={time_bf16:.3f}s")
print("\nNote: On CPU, there's no speedup (or it may be slower). On GPU with Tensor Cores, expect 1.5-3x speedup.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(losses_fp32, label='FP32', linewidth=2)
axes[0].plot(losses_fp16, label='FP16 (AMP)', linewidth=2, linestyle='--')
axes[0].plot(losses_bf16, label='BF16 (AMP)', linewidth=2, linestyle=':')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Memory comparison (theoretical)
param_count = sum(p.numel() for p in DemoNet().parameters())
mem_fp32 = param_count * 4  # 4 bytes per param
mem_fp16 = param_count * 2  # 2 bytes per param (but master weights still FP32)
mem_amp = param_count * 4 + param_count * 2  # master FP32 + FP16 copy

modes = ['FP32\n(weights)', 'AMP\n(master + FP16)', 'FP16 only\n(not recommended)']
mems = [mem_fp32 / 1024, mem_amp / 1024, mem_fp16 / 1024]
colors = ['#2196F3', '#FF9800', '#F44336']
axes[1].bar(modes, mems, color=colors)
axes[1].set_ylabel('Memory (KB)')
axes[1].set_title(f'Parameter Memory ({param_count:,} params)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 4. Memory Savings Breakdown

In AMP training, memory is used by:

| Component | FP32 Training | AMP Training |
|-----------|--------------|-------------|
| Model weights | FP32 | FP32 (master) + FP16 (compute) |
| Activations | FP32 | **FP16** (biggest saving!) |
| Gradients | FP32 | FP16 |
| Optimizer states | FP32 | FP32 (from master weights) |

**Activations dominate memory** for large models. Cutting them from FP32 to FP16 halves activation memory.

In [ ]:
# Demonstrate activation memory savings
batch_size = 32
seq_len = 512
hidden_dim = 768
num_layers = 12

# Approximate activation memory per transformer layer
# attention: QKV projections + attention scores + output
activation_per_layer = (
    batch_size * seq_len * hidden_dim * 4  # QKV + output, each hidden_dim
    + batch_size * 12 * seq_len * seq_len  # attention scores (12 heads)
    + batch_size * seq_len * 4 * hidden_dim  # FFN intermediate
)

total_fp32 = activation_per_layer * num_layers * 4  # 4 bytes
total_fp16 = activation_per_layer * num_layers * 2  # 2 bytes

print(f"Approximate activation memory for BERT-base ({num_layers} layers):")
print(f"  Batch size:  {batch_size}")
print(f"  Seq length:  {seq_len}")
print(f"  Hidden dim:  {hidden_dim}")
print(f"  FP32: {total_fp32 / 1e9:.2f} GB")
print(f"  FP16: {total_fp16 / 1e9:.2f} GB")
print(f"  Savings: {(total_fp32 - total_fp16) / 1e9:.2f} GB ({50}%)")

## 5. Which Ops Are Safe in FP16?

PyTorch autocast automatically handles this.

In [ ]:
print("=== Operations that run in FP16 (safe, big speedup) ===")
print("  - Matrix multiplications (linear layers, conv layers)")
print("  - Batch/Layer normalization")
print("  - Activation functions (ReLU, GELU, etc.)")
print()

print("=== Operations that MUST stay in FP32 ===")
print("  - Loss computation (cross_entropy, mse_loss)")
print("  - Softmax (large values can overflow in FP16)")
print("  - Log, exp, pow (need precision)")
print("  - Reduction operations (sum, mean over large tensors)")
print("  - Batch norm running statistics")
print()

print("=== autocast handles this automatically ===")

# Demonstrate autocast behavior
x = torch.randn(4, 8)
w = torch.randn(8, 8)

print("\nWithout autocast:")
print(f"  matmul dtype: {torch.mm(x, w).dtype}")

# Note: autocast with float16 on CPU may not change dtype on all PyTorch versions;
# the real speedup is on CUDA. This shows the API pattern.
with autocast(device_type='cpu', dtype=torch.bfloat16):
    result = torch.mm(x, w)
    print(f"\nWith autocast (bfloat16):")
    print(f"  matmul dtype: {result.dtype}")
    
    # Loss stays in FP32
    logits = torch.randn(4, 10)
    targets = torch.randint(0, 10, (4,))
    loss = nn.functional.cross_entropy(logits, targets)
    print(f"  cross_entropy dtype: {loss.dtype}")

## 6. Complete AMP Training Template

In [ ]:
def train_with_amp_template(model, train_loader, optimizer, criterion, 
                            device='cpu', dtype=torch.bfloat16, epochs=5):
    """
    Production AMP training template.
    On GPU: device='cuda', dtype=torch.float16 or torch.bfloat16
    """
    model = model.to(device)
    use_scaler = (dtype == torch.float16 and device != 'cpu')
    scaler = GradScaler(device, enabled=use_scaler)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            optimizer.zero_grad(set_to_none=True)  # slightly more efficient
            
            # Forward pass in mixed precision
            with autocast(device_type=device, dtype=dtype):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            
            # Backward pass with gradient scaling
            scaler.scale(loss).backward()
            
            # Optional: gradient clipping (must unscale first)
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / (batch_idx + 1)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")


# Demo with simple data
from torch.utils.data import TensorDataset, DataLoader

X = torch.randn(500, 784)
y = torch.randint(0, 10, (500,))
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

model = DemoNet()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

train_with_amp_template(model, loader, optimizer, criterion, 
                        device='cpu', dtype=torch.bfloat16, epochs=5)

## Interview Questions & Answers

---

**Q: Why BF16 over FP16?**

BF16 has the same exponent range as FP32 (8 exponent bits), so:
- No overflow (FP16 max is 65504, BF16 max is ~3e38)
- No gradient underflow for small values
- No loss scaling needed (simpler training)
- Less precise (7-bit mantissa vs 10-bit), but this rarely matters for training

Most modern hardware (A100, H100, TPU) supports BF16 natively. **Use BF16 if your hardware supports it.**

---

**Q: What is loss scaling?**

FP16 can't represent values smaller than ~6e-8. Many gradients are in this range and become zero. Loss scaling:
1. Multiply loss by a large scale factor S before backward()
2. All gradients are also scaled by S (chain rule)
3. Divide gradients by S before optimizer.step()

Dynamic scaling: start large, halve on inf/nan, double periodically. GradScaler does this automatically.

---

**Q: Which ops are safe in FP16?**

Safe (and fast): matmul, linear, conv, activation functions, batch/layer norm.  
Must stay FP32: softmax, cross-entropy, log/exp, reductions over large tensors.  
`autocast` handles this automatically.

---

**Q: Typical speedup numbers?**

- NVIDIA A100: 1.5-3x speedup with Tensor Cores
- Memory: ~50% reduction in activation memory
- Throughput: can often double batch size
- Accuracy: no degradation for almost all models

---

**Q: Does AMP always help?**

- Needs hardware with FP16/BF16 acceleration (Tensor Cores, TPU)
- Small models may not see speedup (overhead of casting dominates)
- Memory savings are always real, even if speed doesn't improve much

## Quick Reference

```python
# Minimal AMP pattern
scaler = GradScaler('cuda')  # only needed for FP16

for batch in loader:
    optimizer.zero_grad(set_to_none=True)
    
    with autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = model(batch)
    
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)  # if doing grad clipping
    clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
```